# 05. Análisis del Dataset

Una vez que hemos limpiado y validado los datos, podemos comenzar a analizarlos con Pandas.

En este apartado trabajaremos con el dataset limpio para obtener métricas, comparar categorías y analizar la evolución de los datos en el tiempo.

---

## 5.1 Cargar el dataset limpio

Antes de iniciar el análisis, cargamos el archivo que hemos guardado en `data/processed/`.

```python
import pandas as pd

df = pd.read_csv(
    "../data/processed/dataset_limpio.csv",
    parse_dates=["fecha"]
)

df.head()
```

- `pd.read_csv()` carga el archivo CSV y devuelve un DataFrame.
- `parse_dates=["fecha"]` indica a Pandas que debe interpretar la columna `fecha` como una columna de fechas durante la lectura.
- `df.head()` permite comprobar que los datos se han cargado correctamente.

**Importante:** el nombre `fecha` debe coincidir exactamente con el nombre de la columna del archivo. Si el dataset no contiene esa columna, debemos quitar `parse_dates` o indicar la columna de fecha correcta.

Podemos comprobar los tipos de datos:

```python
df.dtypes
```

La columna `fecha` debería aparecer con un tipo de dato de fecha, normalmente `datetime64[ns]`.

---

## 5.2 Análisis básico

Podemos calcular métricas generales sobre una columna numérica.

Por ejemplo, para conocer el precio medio:

```python
df["precio"].mean()
```

`mean()` calcula la media aritmética de los valores no nulos de la columna.

También podemos calcular otras métricas habituales:

```python
df["precio"].sum()   # Suma de los precios.
df["precio"].min()   # Precio mínimo.
df["precio"].max()   # Precio máximo.
df["precio"].count() # Número de valores no nulos.
```

Estas operaciones permiten obtener una primera visión de los datos. Antes de interpretar los resultados, debemos tener claro qué representa cada registro y qué unidad tiene la columna analizada.

---

## 5.3 Análisis por categoría

Podemos agrupar los registros por una categoría y calcular una métrica para cada grupo.

Por ejemplo, para conocer el importe total por categoría y ordenar los resultados de mayor a menor:

```python
df.groupby("categoria")["precio"].sum().sort_values(ascending=False)
```

El proceso es el siguiente:

1. `groupby("categoria")` agrupa los registros según los valores de la columna `categoria`.
2. `["precio"]` selecciona la columna sobre la que queremos realizar el cálculo.
3. `.sum()` suma los precios de cada grupo.
4. `.sort_values(ascending=False)` ordena los resultados de mayor a menor.

El resultado muestra el total de la columna `precio` para cada categoría.

**Nota:** `sum()` no calcula necesariamente ventas o ingresos. Para interpretar el resultado como facturación, debemos comprobar que `precio` representa realmente el importe vendido por registro. Si hay cantidades, descuentos o varias unidades, quizá debamos analizar otra columna, como `importe` o `total_venta`.

---

## 5.4 Varias métricas con `agg()`

Cuando necesitamos calcular varias métricas de una misma columna dentro de cada grupo, podemos utilizar `agg()`.

```python
df.groupby("categoria").agg(
    media_precio=("precio", "mean"),
    max_precio=("precio", "max"),
    num_registros=("precio", "count")
)
```

En este ejemplo calculamos:

- `media_precio`: precio medio de cada categoría.
- `max_precio`: precio máximo de cada categoría.
- `num_registros`: número de valores no nulos de `precio` en cada categoría.

La sintaxis de cada métrica es:

```python
nombre_resultado=("columna", "funcion")
```

Podemos asignar nombres descriptivos a las columnas del resultado, lo que facilita su lectura y posterior utilización.

**Importante sobre `count()`:** cuenta los valores no nulos de la columna indicada. Si queremos contar todas las filas de cada grupo, independientemente de que `precio` tenga valores nulos, podemos utilizar:

```python
df.groupby("categoria").size()
```

Por tanto:

- `count()` cuenta valores no nulos de una columna.
- `size()` cuenta filas de cada grupo.

---

## 5.5 Análisis temporal

También podemos agrupar los datos por fecha para analizar la evolución de una métrica.

Si queremos obtener el importe total de cada día:

```python
df_por_dia = (
    df.groupby(df["fecha"].dt.normalize())["precio"]
    .sum()
)

df_por_dia
```

En este caso:

1. `df["fecha"].dt.normalize()` conserva la fecha y establece la hora a `00:00:00`.
2. `groupby()` agrupa los registros que pertenecen al mismo día.
3. `["precio"].sum()` suma los precios de cada día.
4. Guardamos el resultado en `df_por_dia`.

**¿Por qué no agrupamos directamente por `fecha`?**

Si la columna contiene fecha y hora (`datetime`), dos registros del mismo día podrían tener valores distintos porque sus horas son diferentes. Al normalizar la fecha, todos los registros del mismo día quedan agrupados juntos.

Si la columna ya contiene únicamente fechas, sin horas, también podemos utilizar directamente:

```python
df.groupby("fecha")["precio"].sum()
```

---

## Resumen

En este apartado hemos trabajado con operaciones habituales para analizar un dataset:

| Operación | Utilidad |
|---|---|
| `pd.read_csv()` | Cargar el dataset limpio. |
| `parse_dates` | Interpretar columnas como fechas al cargar el archivo. |
| `mean()` | Calcular la media. |
| `sum()` | Calcular la suma. |
| `min()` / `max()` | Obtener el mínimo y el máximo. |
| `count()` | Contar valores no nulos. |
| `groupby()` | Agrupar registros según una o varias columnas. |
| `agg()` | Calcular varias métricas por grupo. |
| `sort_values()` | Ordenar los resultados. |
| `.dt.normalize()` | Agrupar fechas ignorando la hora. |

El objetivo no es únicamente ejecutar operaciones, sino entender qué pregunta de negocio responde cada cálculo y comprobar que la métrica utilizada representa correctamente lo que queremos analizar.
